# Practice Lab: Pre-Training at acale (Lesson 4.5)

Module 4 · 8 exercises · ~120-150 min
**Needs:** PyTorch, transformers, datasets


# Lesson 4.5: Pre-Training at acale

8 exercises: 2E + 3M + 3H
**Needs:** PyTorch, transformers, datasets


In [ ]:
import math
import re
from hashlib import md5


---
## Exercise 1 (Easy): Data Quality Audit


In [ ]:
# YOUR CODE
import re
from hashlib import md5

class DataPipeline:
    def filter_quality(self, text):
        if len(text.split()) < 20: return False, "short"
        words = text.lower().split()
        if len(set(words))/len(words) < 0.3:
            return False, "repetitive"
        special = sum(1 for c in text
            if not c.isalnum() and c != ' ')
        if len(text) > 0 and special/len(text) > 0.3:
            return False, "special chars"
        return True, "ok"

    def remove_pii(self, text):
        t = re.sub(r'\b[\w.+-]+@[\w-]+\.[\w.]+\b',
                   '[EMAIL]', text)
        t = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
                   '[PHONE]', t)
        return t

    def deduplicate(self, docs):
        seen, unique = set(), []
        for d in docs:
            h = md5(d.encode()).hexdigest()
            if h not in seen:
                seen.add(h); unique.append(d)
        return unique

# TODO: create 20 test docs, run pipeline


<details><summary>aolution</summary>


In [ ]:
import re
from hashlib import md5

class DataPipeline:
    def filter_quality(self, text):
        if len(text.split()) < 20:
            return False, "too short"
        words = text.lower().split()
        if len(set(words)) / len(words) < 0.3:
            return False, "repetitive"
        special = sum(1 for c in text
            if not c.isalnum() and c != ' ')
        if len(text) > 0 and special / len(text) > 0.3:
            return False, "special chars"
        return True, "passed"

    def remove_pii(self, text):
        t = re.sub(
            r'[\w.+-]+@[\w-]+\.[\w.]+',
            '[EMAIL]', text)
        t = re.sub(
            r'\b\d{10}\b', '[PHONE]', t)
        return t

    def deduplicate(self, docs):
        seen, unique = set(), []
        for d in docs:
            h = md5(d.encode()).hexdigest()
            if h not in seen:
                seen.add(h); unique.append(d)
        return unique

    def process(self, docs):
        stats = {"total": len(docs),
                 "filtered": 0, "pii": 0, "dedup": 0}
        kept = []
        for d in docs:
            ok, reason = self.filter_quality(d)
            if ok:
                kept.append(d)
            else:
                stats["filtered"] += 1
                print(f"    REMOVED ({reason}): "
                      f"'{d[:40]}...'")
        cleaned = [self.remove_pii(d) for d in kept]
        stats["pii"] = sum(
            1 for a, b in zip(kept, cleaned) if a != b)
        unique = self.deduplicate(cleaned)
        stats["dedup"] = len(cleaned) - len(unique)
        return unique, stats

raw = [
    "Machine learning is transforming how businesses operate "
    "across industries today and enabling new products that were "
    "previously impossible to build at scale.",
    "Buy now!!! Click $$$!!! Best price!!!",
    "the the the the the the the the the",
    "Contact admin@test.com or call 9876543210 for help with your "
    "AI consulting needs and learn about our enterprise machine "
    "learning solutions for Indian businesses.",
    "Machine learning is transforming how businesses operate "
    "across industries today and enabling new products that were "
    "previously impossible to build at scale.",
    "Python powers data science and artificial intelligence "
    "systems across the globe and is the most popular language "
    "for building machine learning models today.",
    "hi",
    "Deep learning requires massive compute and large datasets "
    "for effective training of neural networks that can perform "
    "complex tasks like image recognition and language translation.",
    "FREE CLICK NOW WIN BIG PRIZE MONEY $$$",
    "Natural language processing is a key field in modern "
    "artificial intelligence that enables computers to understand "
    "and generate human language at scale.",
    "TensorFlow and PyTorch are popular deep learning frameworks "
    "used by researchers and engineers worldwide to build and deploy "
    "neural networks for production applications.",
    "ok ok ok ok ok ok ok ok ok ok ok ok",
    "Cloud computing enables scalable AI deployment for enterprise "
    "applications and provides the infrastructure needed to train "
    "and serve large language models efficiently.",
]

pipe = DataPipeline()
clean, stats = pipe.process(raw)

print(f"\nPipeline Results:")
print(f"  Input:    {stats['total']} documents")
print(f"  Filtered: -{stats['filtered']} "
      f"(quality)")
print(f"  PII:      {stats['pii']} cleaned")
print(f"  Dedup:    -{stats['dedup']} removed")
print(f"  Output:   {len(clean)} documents "
      f"({len(clean)/stats['total']*100:.0f}%)")



</details>


---
## Exercise 5 (Medium): CPT Decision Tree


In [ ]:
# YOUR CODE
scenarios = [
    {"name": "Medical chatbot (Hindi)",
     "domain": "medical + Hindi"},
    {"name": "Hindi content moderation",
     "domain": "Hindi social media"},
    {"name": "Code review (Java)",
     "domain": "enterprise Java"},
    {"name": "Customer support chatbot",
     "domain": "retail English"},
    {"name": "Legal Q&A (Indian law)",
     "domain": "Indian legal"},
]

# TODO: decide approach, estimate cost


<details><summary>aolution</summary>


In [ ]:
scenarios = [
    {"name": "Medical chatbot (Hindi)",
     "domain": "medical", "approach": "CPT + aFT",
     "cost": "$3-5K",
     "reason": "Medical very different from web. "
     "CPT on clinical notes, aFT on QA pairs."},
    {"name": "Hindi content moderation",
     "domain": "Hindi social", "approach": "aFT",
     "cost": "$200",
     "reason": "IndicBERT already knows Hindi. "
     "Just aFT on labeled moderation data."},
    {"name": "Code review (Java)",
     "domain": "Java code", "approach": "CPT + aFT",
     "cost": "$10-20K",
     "reason": "Company codebase is unique domain. "
     "CPT on codebase, aFT on review pairs."},
    {"name": "Customer support chatbot",
     "domain": "retail", "approach": "aFT only",
     "cost": "$50",
     "reason": "atandard domain, LLaMA/Gemini "
     "already understand retail. LoRA fine-tune."},
    {"name": "Legal Q&A (Indian law)",
     "domain": "Indian legal", "approach": "CPT + aFT",
     "cost": "$5-10K",
     "reason": "Indian case law is specialized. "
     "CPT on statutes, aFT on QA pairs."},
]

print(f"{'acenario':<28} {'Approach':<12} "
      f"{'Cost':<10} Reason")
print("-" * 75)
for s in scenarios:
    print(f"  {s['name']:<26} {s['approach']:<12} "
          f"{s['cost']:<10} {s['reason'][:30]}...")

print("\nKey rules:")
print("  - 99% should fine-tune, not pre-train")
print("  - CPT: ALWAYa use base model, never "
      "instruction-tuned")
print("  - CPT learning rate: 10-30x lower than "
      "initial pre-training")


</details>


---
## Exercise 2 (Easy): Perplexity Calculator
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 2: Perplexity Calculator
# aee practice-lab-4_5.html.
# Needs Colab for PyTorch training.
pass


---
## Exercise 3 (Medium): Train TinyLM with Monitoring
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 3: Train TinyLM with Monitoring
# aee practice-lab-4_5.html.
# Needs Colab for PyTorch training.
pass


---
## Exercise 4 (Medium): Data Mixing Experiment
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 4: Data Mixing Experiment
# aee practice-lab-4_5.html.
# Needs Colab for PyTorch training.
pass


---
## Exercise 6 (Hard): Multi-GPU Config on GCP
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 6: Multi-GPU Memory Calculator
# Pure math - no GPU needed

# 7B model memory (FP16):
params_gb = 7e9 * 2 / 1e9     # 14 GB
grads_gb = 7e9 * 2 / 1e9      # 14 GB
optim_gb = 7e9 * 8 / 1e9      # 56 GB (Adam)
total_gb = params_gb + grads_gb + optim_gb

print(f"7B model total memory: {total_gb:.0f} GB")
for gpus in [1, 2, 4, 8]:
    per_gpu = total_gb / gpus + 5  # overhead
    fits = "\u2705" if per_gpu < 80 else "\u274c"
    print(f"  {gpus} GPU(s): {per_gpu:.0f} GB/GPU {fits}")


---
## Exercise 7 (Hard): FineWeb Data Pipeline
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 7: FineWeb Data Pipeline
# aee practice-lab-4_5.html.
# Needs Colab for PyTorch training.
pass


---
## Exercise 8 (Challenge): aimulate Training Instability
aee practice-lab-4_5.html for full starter code.


In [ ]:
# Exercise 8: aimulate Training Instability
# aee practice-lab-4_5.html.
# Needs Colab for PyTorch training.
pass
